# 02 — Cleaning & Exploratory Data Analysis

Turns the raw CSVs from notebook 01 into a clean, queryable DuckDB database, then
explores it. The notebook spans two build phases:

- **Phase 2 — cleaning & schema.** Load raw CSVs into typed DuckDB tables, standardise
  on UTC, melt the wide ENTSO-E generation feed to long format, and resample the
  15-minute series to hourly.
- **Phase 3 — EDA.** Verify completeness, profile distributions, and surface the
  seasonal patterns each research question later interrogates.

| Phase | What | Output |
|---|---|---|
| 2 | CSV → typed tables · UTC · wide→long · 15-min→hourly | clean DuckDB tables |
| 3 | completeness · distributions · seasonal & diurnal profiles | QA tables + EDA figures |

**Database:** `data/processed/austria_energy.duckdb`. All timestamps stored as UTC
`TIMESTAMPTZ`; converted to `Europe/Vienna` only at the display layer.

**Prerequisite:** `01_data_collection.ipynb` has been run — raw CSVs present in `data/raw/`.

In [ ]:
import duckdb
from pathlib import Path
import pandas as pd

In [ ]:
# Cell A: inspect raw CSVs before schema design

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"

files = [
    "entsoe_generation_AT.csv",
    "entsoe_load_AT.csv",
    "entsoe_prices_AT.csv",
    "weather_vienna.csv",
]

for fname in files:
    path = RAW / fname
    if not path.exists():
        print(f"!! MISSING: {path}\n")
        continue
    size_kb = path.stat().st_size / 1024
    print("=" * 80)
    print(f"  {fname}   ({size_kb:,.0f} KB)")
    print("=" * 80)
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            print(f"L{i}: {line.rstrip()[:160]}")
    print()

In [ ]:
# Cell B: open DuckDB + create schema

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

DB_PATH = PROCESSED / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH))

# Clean slate — drop everything for safe re-runs while iterating
for tbl in ["generation_15min", "demand_15min", "generation",
            "demand", "prices", "weather", "owid_energy_at", "ghg_emissions",
            "esr_emissions"]:
    con.execute(f"DROP TABLE IF EXISTS {tbl}")

## This version is just for DuckDB/PostgreSQL dialect
## For example, for generation_15min, the following is the correct syntax:

## con.execute("""
## CREATE OR REPLACE TABLE generation_15min (
##     ts_utc      TIMESTAMPTZ NOT NULL,
##     fuel_type   VARCHAR     NOT NULL,
##     flow        VARCHAR     NOT NULL,
##     mw          DOUBLE,
##     PRIMARY KEY (ts_utc, fuel_type, flow)
## )
## """)

# ── Staging (15-min, long format) ────────────────────────────────────────
con.execute("""
CREATE TABLE generation_15min (
    ts_utc      TIMESTAMPTZ NOT NULL,
    fuel_type   VARCHAR     NOT NULL,
    flow        VARCHAR     NOT NULL,      -- 'generation' or 'consumption'
    mw          DOUBLE,
    PRIMARY KEY (ts_utc, fuel_type, flow)
)
""")

con.execute("""
CREATE TABLE demand_15min (
    ts_utc     TIMESTAMPTZ PRIMARY KEY,
    demand_mw  DOUBLE          -- allow NULL; we'll count gaps in QA
)
""")

# ── Final hourly tables (no resampling needed — native hourly) ──────────
con.execute("""
CREATE TABLE prices (
    ts_utc         TIMESTAMPTZ PRIMARY KEY,
    price_eur_mwh  DOUBLE       -- allow NULL; we'll count gaps in QA
)
""")

con.execute("""
CREATE TABLE weather (
    ts_utc          TIMESTAMPTZ PRIMARY KEY,
    temperature_c   DOUBLE,
    solar_wm2       DOUBLE,
    wind_kmh        DOUBLE,
    precip_mm       DOUBLE,
    cloudcover_pct  DOUBLE
)
""")

# generation, demand, owid_energy_at are created via CTAS in later cells.

print(con.sql("SHOW TABLES").df())

In [ ]:
# Cell C: load OWID + prices + weather

RAW = PROJECT_ROOT / "data" / "raw"

# ── 1. OWID — annual, no TZ, CTAS straight from CSV ─────────────────────
con.execute(f"""
CREATE TABLE owid_energy_at AS
SELECT *
FROM read_csv_auto('{RAW / "owid_energy_AUT.csv"}', header=true)
""")
n = con.sql("SELECT COUNT(*) FROM owid_energy_at").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"owid_energy_at: {n} rows")

# ── 2. Prices — timestamps already carry offsets, parse to UTC ──────────
df_p = pd.read_csv(RAW / "entsoe_prices_AT.csv", index_col=0, parse_dates=True)
df_p.index = pd.to_datetime(df_p.index, utc=True)         # to UTC
df_p = df_p.rename_axis("ts_utc").reset_index()
df_p.columns = ["ts_utc", "price_eur_mwh"]
# there exists an explicit form of insert
# It is for assuring to be name sensitive about columns and not just order
con.execute("INSERT INTO prices SELECT * FROM df_p")
n = con.sql("SELECT COUNT(*) FROM prices").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"prices: {n} rows")

# ── 3. Weather — already UTC, just tag the index ────────────────────────
df_w = pd.read_csv(RAW / "weather_vienna.csv",
                   index_col="timestamp", parse_dates=True)
df_w.index = df_w.index.tz_localize("UTC")     # naive → UTC label, no shift  # pyright: ignore[reportAttributeAccessIssue]
df_w = df_w.rename(columns={
    "temperature_2m":      "temperature_c",
    "shortwave_radiation": "solar_wm2",
    "windspeed_10m":       "wind_kmh",
    "precipitation":       "precip_mm",
    "cloudcover":          "cloudcover_pct",
}).rename_axis("ts_utc").reset_index()
con.execute("INSERT INTO weather SELECT * FROM df_w")
n = con.sql("SELECT COUNT(*) FROM weather").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"weather: {n} rows")

In [ ]:
# Cell D: sanity check what's in the DB - Part 1
con.sql("""
SELECT 'prices'  AS tbl, COUNT(*) AS rows,
       MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM prices
UNION ALL
SELECT 'weather', COUNT(*), MIN(ts_utc), MAX(ts_utc) FROM weather
UNION ALL
SELECT 'owid',    COUNT(*), NULL, NULL FROM owid_energy_at
""").df()

In [ ]:
# Cell D: sanity check what's in the DB - Part 2
con.sql("""
SELECT 'demand_15min' AS tbl,
       COUNT(*)                              AS rows,
       COUNT(*) FILTER (WHERE demand_mw IS NULL) AS nulls,
       ROUND(100.0 * COUNT(*) FILTER (WHERE demand_mw IS NULL) / COUNT(*), 3)
                                             AS pct_null
FROM demand_15min
UNION ALL
SELECT 'generation_15min',
       COUNT(*),
       COUNT(*) FILTER (WHERE mw IS NULL),
       ROUND(100.0 * COUNT(*) FILTER (WHERE mw IS NULL) / COUNT(*), 3)
FROM generation_15min
""").df()

In [ ]:
# Cell E: melt generation wide → long, load demand

# ── 4. Generation — wide MultiIndex CSV → long staging ─────────────────────
df_g = pd.read_csv(
    RAW / "entsoe_generation_AT.csv",
    header=[0, 1],          # 2-row header → MultiIndex columns
    index_col=0,
)
# Mixed offsets across DST → utc=True normalises both to UTC
df_g.index = pd.to_datetime(df_g.index, utc=True)
df_g.index.name = "ts_utc"

# Name the column levels so stack() produces nice column names
df_g.columns.names = ["fuel_type", "_flow_raw"]

# Wide → long: every (fuel_type, flow) column becomes its own row
df_g_long = (
    df_g
    .stack(level=["fuel_type", "_flow_raw"], future_stack=True)
    .rename("mw")  # pyright: ignore[reportArgumentType]
    .reset_index()
    .dropna(subset=["mw"])      # drop rows ENTSO-E didn't report
)

# Map ENTSO-E flow labels → our schema
flow_map = {
    "Actual Aggregated":  "generation",
    "Actual Consumption": "consumption",
}
df_g_long["flow"] = df_g_long["_flow_raw"].map(flow_map)

# Defensive: error if ENTSO-E ships a new flow label we haven't handled
unmapped = df_g_long["flow"].isna()
if unmapped.any():
    raise ValueError(
        f"Unknown ENTSO-E flow types: "
        f"{df_g_long.loc[unmapped, '_flow_raw'].unique().tolist()}"
    )

df_g_long = df_g_long[["ts_utc", "fuel_type", "flow", "mw"]]

con.execute("""
    INSERT INTO generation_15min (ts_utc, fuel_type, flow, mw)
    SELECT ts_utc, fuel_type, flow, mw FROM df_g_long
""")
n = con.sql("SELECT COUNT(*) FROM generation_15min").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"generation_15min: {n:,} rows")


# ── 5. Demand — single-column CSV, straight-through ────────────────────────
df_d = pd.read_csv(
    RAW / "entsoe_load_AT.csv",
    index_col=0,
)
df_d.index = pd.to_datetime(df_d.index, utc=True)
df_d = df_d.rename_axis("ts_utc").reset_index()
df_d.columns = ["ts_utc", "demand_mw"]

con.execute("""
    INSERT INTO demand_15min (ts_utc, demand_mw)
    SELECT ts_utc, demand_mw FROM df_d
""")
n = con.sql("SELECT COUNT(*) FROM demand_15min").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"demand_15min: {n:,} rows")

In [ ]:
# Cell F: re-run QA on the populated tables
con.sql("""
SELECT 'generation_15min' AS tbl,
       COUNT(*)                                  AS rows,
       COUNT(DISTINCT fuel_type)                 AS n_fuels,
       COUNT(*) FILTER (WHERE mw IS NULL)        AS nulls,
       MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM generation_15min
UNION ALL
SELECT 'demand_15min', COUNT(*), NULL,
       COUNT(*) FILTER (WHERE demand_mw IS NULL),
       MIN(ts_utc), MAX(ts_utc)
FROM demand_15min
""").df()

In [ ]:
# Cell G: resample 15-min → hourly via SQL

# ── 6. Generation hourly: filter junk consumption rows + aggregate ─────────
con.execute("""
CREATE OR REPLACE TABLE generation AS
SELECT
    time_bucket(INTERVAL 1 HOUR, ts_utc) AS ts_utc,
    fuel_type,
    flow,
    AVG(mw)   AS mw,            -- MW is instantaneous power → average, don't sum
    COUNT(*)  AS n_quarters     -- expect 4 per row; <4 means source gap
FROM generation_15min
WHERE flow = 'generation'                                       -- always keep
   OR (flow = 'consumption' AND fuel_type = 'Hydro Pumped Storage')
                                                                -- only pumped storage actually consumes
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
""")
n_g = con.sql("SELECT COUNT(*) FROM generation").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"generation: {n_g:,} rows")

# ── 7. Demand hourly ────────────────────────────────────────────────────────
con.execute("""
CREATE OR REPLACE TABLE demand AS
SELECT
    time_bucket(INTERVAL 1 HOUR, ts_utc) AS ts_utc,
    AVG(demand_mw)  AS demand_mw,
    COUNT(*)        AS n_quarters
FROM demand_15min
GROUP BY 1
ORDER BY 1
""")
n_d = con.sql("SELECT COUNT(*) FROM demand").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"demand: {n_d:,} rows")

In [ ]:
# Quarter-count check: should always be 4
display(con.sql("""
SELECT 'generation' AS tbl, n_quarters, COUNT(*) AS n_rows
FROM generation
GROUP BY n_quarters
UNION ALL
SELECT 'demand', n_quarters, COUNT(*)
FROM demand
GROUP BY n_quarters
ORDER BY tbl, n_quarters
""").df())

# What's actually in Austria's generation mix?
display(con.sql("""
SELECT
    fuel_type,
    flow,
    ROUND(AVG(mw))      AS avg_mw,
    ROUND(MAX(mw))      AS peak_mw,
    ROUND(SUM(mw) / 1e6, 2) AS total_twh   -- MW × hours / 1e6 = TWh
FROM generation
GROUP BY fuel_type, flow
ORDER BY avg_mw DESC NULLS LAST
""").df())

In [ ]:
con.sql("""
SELECT fuel_type,
       COUNT(DISTINCT mw)         AS n_unique_values,
       ROUND(STDDEV(mw), 4)       AS std_mw,
       ROUND(MIN(mw), 2)          AS min_mw,
       ROUND(MAX(mw), 2)          AS max_mw
FROM generation
WHERE flow = 'generation'
GROUP BY fuel_type
ORDER BY n_unique_values
""").df()

## Phase 2 complete

DuckDB file: `data/processed/austria_energy.duckdb`

| Table | Grain | Source |
|---|---|---|
| `generation`        | hourly × (fuel_type, flow) | ENTSO-E (resampled from 15-min) |
| `demand`            | hourly                     | ENTSO-E (resampled from 15-min) |
| `prices`            | hourly                     | ENTSO-E day-ahead              |
| `weather`           | hourly                     | Open-Meteo / ERA5 (Vienna)     |
| `owid_energy_at`    | annual                     | Our World in Data              |

All timestamps stored as UTC TIMESTAMPTZ. Staging tables (`generation_15min`,
`demand_15min`) retained for reproducibility. Resampling: `AVG(mw)` over
`time_bucket(INTERVAL 1 HOUR, ts_utc)`.

Notable: hydro dominates Austria's mix; coal phased out April 2020;
solar peak-to-avg ratio ~19× drives the duck curve dynamics.

## Phase 3 — Exploratory Data Analysis

Clean tables in hand, the work shifts from *shaping* the data to *understanding* it.
Three passes over the hourly electricity data, plus a first look at the annual series —
each finding a hook for a later research question.

| Pass | What | Feeds |
|---|---|---|
| Completeness | gaps & nulls, proven with a window-function scan | trust in every RQ |
| Distributions | robust summary stats first; plot only what a number can't show | RQ4 (price regime) |
| Seasonal & diurnal | demand by hour/weekday/month, solar summer-vs-winter, temp arc | RQ2, RQ3 |
| Annual first-look | GHG trajectory + sector decomposition | RQ6 |

Two principles carried throughout:

- **Numbers before pictures.** A summary table that says everything makes a plot
  redundant; a figure earns its place only when it shows something a statistic can't
  (here, the 2022 price regime shift).
- **Local time only at the edge.** Every clock-dependent aggregation converts
  `ts_utc → Europe/Vienna` at query time — a diurnal profile on raw UTC is shifted 1–2 h
  and smears the DST fall-back hour.

This phase also introduces the shared plotting helpers in `src/viz.py`
(`set_house_style()`, `PALETTE`, `line_profile()`).

In [ ]:
# Cell H (Phase 3): missingness done right

# Continuity: in a time series, a missing hour is an ABSENT ROW, not a null.
# expected_hours = span in hours + 1 (inclusive). If actual == expected and
# there are no gaps, the index is continuous.
# NOTE: f-string SQL here is fine because table/column names are hardcoded
# constants — never do this with user-supplied input (SQL injection).
def continuity_check(con, table: str, value_col: str) -> pd.DataFrame:
    return con.sql(f"""
        SELECT
            '{table}'                                          AS tbl,
            COUNT(*)                                           AS rows,
            date_diff('hour', MIN(ts_utc), MAX(ts_utc)) + 1    AS expected_hours,
            date_diff('hour', MIN(ts_utc), MAX(ts_utc)) + 1
                - COUNT(*)                                     AS missing_hours,
            COUNT(*) FILTER (WHERE {value_col} IS NULL)        AS null_values
        FROM {table}
    """).df()

checks = [("demand", "demand_mw"), ("prices", "price_eur_mwh")]
display(pd.concat([continuity_check(con, t, c) for t, c in checks],
                  ignore_index=True))

# Weather has 5 value columns — scan each
display(con.sql("""
SELECT
    COUNT(*)                                         AS rows,
    date_diff('hour', MIN(ts_utc), MAX(ts_utc)) + 1  AS expected_hours,
    COUNT(*) FILTER (WHERE temperature_c  IS NULL)   AS null_temp,
    COUNT(*) FILTER (WHERE solar_wm2      IS NULL)   AS null_solar,
    COUNT(*) FILTER (WHERE wind_kmh       IS NULL)   AS null_wind,
    COUNT(*) FILTER (WHERE precip_mm      IS NULL)   AS null_precip,
    COUNT(*) FILTER (WHERE cloudcover_pct IS NULL)   AS null_cloud
FROM weather
""").df())

In [ ]:
# Cell H (cont.): explicitly locate any gap using a window function.
# LAG pulls the previous row's timestamp; any step ≠ 1 hour is a gap.
# (This is your first window function — the OVER(ORDER BY) pattern is the
#  backbone of Phase 4's duck-curve and merit-order analyses.)
con.sql("""
WITH stepped AS (
    SELECT
        ts_utc,
        ts_utc - LAG(ts_utc) OVER (ORDER BY ts_utc) AS step
    FROM demand
)
SELECT ts_utc AS gap_after, step
FROM stepped
WHERE step <> INTERVAL 1 HOUR     -- first row's step is NULL, auto-excluded
ORDER BY ts_utc
""").df()   # expect 0 rows

Continuity verified in UTC, no imputation needed

In [ ]:
# Cell I (Phase 3): summary stats BEFORE any plot
# median + p01/p99 are robust to the 2022 price spike in a way mean/std aren't.
# pct_zero is the zero-inflation flag (watch solar & precip light up).
con.sql("""
WITH series AS (
    SELECT 'demand_mw'  AS metric, demand_mw     AS v FROM demand
    UNION ALL SELECT 'price_eur_mwh', price_eur_mwh FROM prices
    UNION ALL SELECT 'temp_c',        temperature_c  FROM weather
    UNION ALL SELECT 'solar_wm2',     solar_wm2      FROM weather
    UNION ALL SELECT 'wind_kmh',      wind_kmh       FROM weather
    UNION ALL SELECT 'precip_mm',     precip_mm      FROM weather
    UNION ALL SELECT 'cloud_pct',     cloudcover_pct FROM weather
)
SELECT
    metric,
    ROUND(AVG(v), 1)                       AS mean,
    ROUND(median(v), 1)                    AS p50,
    ROUND(stddev(v), 1)                    AS std,
    ROUND(MIN(v), 1)                       AS min,
    ROUND(quantile_cont(v, 0.99), 1)       AS p99,
    ROUND(MAX(v), 1)                       AS max,
    ROUND(skewness(v), 2)                  AS skew,
    ROUND(100.0 * COUNT(*) FILTER (WHERE v = 0) / COUNT(*), 1) AS pct_zero
FROM series
GROUP BY metric
ORDER BY metric
""").df()

In [ ]:
# Cell I (cont.): generation per fuel — the 8 REAL fuels only.
# Excludes the 5 nominal/placeholder fuels (constant or zero output).
# peak_to_avg surfaces solar's ~19× ratio → this is the RQ3 duck-curve hook.
con.sql("""
SELECT
    fuel_type,
    ROUND(AVG(mw))                              AS mean_mw,
    ROUND(median(mw))                           AS p50_mw,
    ROUND(MIN(mw))                              AS min_mw,
    ROUND(MAX(mw))                              AS max_mw,
    ROUND(100.0 * COUNT(*) FILTER (WHERE mw = 0) / COUNT(*), 1) AS pct_zero,
    ROUND(MAX(mw) / NULLIF(AVG(mw), 0), 1)      AS peak_to_avg
FROM generation
WHERE flow = 'generation'
  AND fuel_type NOT IN ('Waste','Other','Geothermal','Fossil Oil','Other renewable')
GROUP BY fuel_type
ORDER BY mean_mw DESC
""").df()

In [ ]:
# Floor integrity check. A hard floor means: min == floor, a pile-up AT the
# floor, and ZERO observations below it.
# Note: comparing a DOUBLE with a tolerance (<= -499.99) instead of `= -500.0`
# is the safe habit — float equality is fragile in general, even though -500
# happens to be exactly representable here.
con.sql("""
SELECT
    MIN(price_eur_mwh)                                AS min_price,
    COUNT(*) FILTER (WHERE price_eur_mwh <= -499.99)  AS at_floor,
    COUNT(*) FILTER (WHERE price_eur_mwh <  -500.01)  AS below_floor,   -- expect 0
    COUNT(*) FILTER (WHERE price_eur_mwh <  0)         AS negative_hours
FROM prices
""").df()

In [ ]:
# Floor hours in Vienna local time — they should cluster in sunny/windy,
# low-demand hours (spring/summer weekend middays). That's the merit-order
# floor: renewable oversupply with nowhere to go.
con.sql("""
SELECT
    ts_utc AT TIME ZONE 'Europe/Vienna'  AS ts_local,
    price_eur_mwh
FROM prices
WHERE price_eur_mwh <= -499.99
ORDER BY ts_local
""").df()

In [ ]:
# When do negative prices happen? If it's the merit-order effect, they cluster
# at midday (solar oversupply). Hour-of-day computed in Vienna LOCAL time —
# this is the seasonal-aggregation pattern we'll lean on throughout Phase 3/4.
con.sql("""
SELECT
    EXTRACT(hour FROM ts_utc AT TIME ZONE 'Europe/Vienna')  AS hour_local,
    COUNT(*) FILTER (WHERE price_eur_mwh < 0)                AS negative_hours,
    ROUND(AVG(price_eur_mwh), 1)                             AS avg_price
FROM prices
GROUP BY hour_local
ORDER BY hour_local
""").df()

In [ ]:
# Cell J (Phase 3): the ONE distribution worth plotting
# Of 13 metrics, only prices show a regime shift a summary number can't convey.
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# make src/ importable from the notebook
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.viz import set_house_style, PALETTE  # noqa: E402

set_house_style()

# Per-year price structure — robust quantiles computed in SQL.
# Year extracted in UTC; the CET offset is immaterial at annual grain.
PROCESSED = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROCESSED / "austria_energy.duckdb"
con = duckdb.connect(str(DB_PATH))

yr = con.sql("""
SELECT
    EXTRACT(year FROM ts_utc AT TIME ZONE 'Europe/Vienna')  AS yr,
    quantile_cont(price_eur_mwh, 0.25)                 AS p25,
    median(price_eur_mwh)                              AS p50,
    quantile_cont(price_eur_mwh, 0.75)                 AS p75,
    quantile_cont(price_eur_mwh, 0.99)                 AS p99,
    MAX(price_eur_mwh)                                 AS max,
    100.0 * COUNT(*) FILTER (WHERE price_eur_mwh < 0)
          / COUNT(*)                                   AS pct_neg
FROM prices
WHERE EXTRACT(year FROM ts_utc AT TIME ZONE 'Europe/Vienna') <= 2024
GROUP BY yr
ORDER BY yr
""").df()

fig, ax = plt.subplots()
x = yr["yr"]

ax.vlines(x, yr["p25"], yr["p75"], color=PALETTE["price"], lw=9, alpha=0.85,
          label="IQR (p25–p75)")
ax.vlines(x, yr["p75"], yr["p99"], color=PALETTE["price"], lw=2, alpha=0.45,
          label="p75 → p99")
ax.scatter(x, yr["p50"], facecolor="white", edgecolor=PALETTE["price"],
           s=45, zorder=5, label="median")
ax.scatter(x, yr["max"], marker="x", color=PALETTE["accent"], s=55, zorder=5,
           label="max (worst single hour)")

ax.axhline(0, color=PALETTE["muted"], lw=0.8)   # negative prices live below this
ax.set_ylabel("day-ahead price (€/MWh)")
ax.set_title("Austria day-ahead prices by year — the 2022 regime shift")
ax.legend(loc="upper left", ncol=2)
plt.show()

display(yr.round(1))

In [ ]:
con.sql("""
SELECT 'prices'  AS tbl, MIN(ts_utc) AS lo, MAX(ts_utc) AS hi, COUNT(*) AS n FROM prices
UNION ALL
SELECT 'demand',         MIN(ts_utc),       MAX(ts_utc),       COUNT(*)       FROM demand
UNION ALL
SELECT 'weather',        MIN(ts_utc),       MAX(ts_utc),       COUNT(*)       FROM weather
ORDER BY tbl
""").df()

In [ ]:
# Cell K (Phase 3): demand seasonality at three grains
# Same recipe each time: convert to Vienna local → EXTRACT a time part → average.
# Local time is non-negotiable here: a diurnal profile on raw UTC is shifted 1–2h.

set_house_style()

def demand_profile(part: str):
    # `part` is a trusted constant (hour/isodow/month) → f-string is safe.
    return con.sql(f"""
        SELECT EXTRACT({part} FROM ts_utc AT TIME ZONE 'Europe/Vienna') AS x,
               AVG(demand_mw) AS avg_demand
        FROM demand
        GROUP BY x ORDER BY x
    """).df()

hour  = demand_profile("hour")
week  = demand_profile("isodow")    # 1 = Mon … 7 = Sun
month = demand_profile("month")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))

ax[0].plot(hour["x"], hour["avg_demand"], color=PALETTE["demand"], marker="o", ms=3)
ax[0].set(title="By hour of day (local)", xlabel="hour", ylabel="avg demand (MW)")

ax[1].plot(week["x"], week["avg_demand"], color=PALETTE["demand"], marker="o", ms=4)
ax[1].set(title="By day of week")
ax[1].set_xticks(range(1, 8)); ax[1].set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])

ax[2].plot(month["x"], month["avg_demand"], color=PALETTE["demand"], marker="o", ms=4)
ax[2].set(title="By month")
ax[2].set_xticks(range(1, 13)); ax[2].set_xticklabels(list("JFMAMJJASOND"))

fig.suptitle("Austrian electricity demand — seasonal profiles", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Cell L (Phase 3): solar diurnal shape, summer vs winter
# The RQ3 duck-curve driver. FILTER (familiar) splits one query into two seasonal
# columns; the CTE converts to local time once.
import matplotlib.pyplot as plt
from src.viz import set_house_style, PALETTE, line_profile
set_house_style()

solar = con.sql("""
WITH solar_local AS (
    SELECT
        ts_utc AT TIME ZONE 'Europe/Vienna' AS loc,   -- convert once, reuse below
        mw
    FROM generation
    WHERE flow = 'generation' AND fuel_type = 'Solar'
)
SELECT
    EXTRACT(hour FROM loc)                                        AS hour_local,
    AVG(mw) FILTER (WHERE EXTRACT(month FROM loc) IN (6, 7, 8))   AS summer_mw,
    AVG(mw) FILTER (WHERE EXTRACT(month FROM loc) IN (12, 1, 2))  AS winter_mw
FROM solar_local
GROUP BY hour_local
ORDER BY hour_local
""").df()

fig, ax = plt.subplots()
line_profile(ax, solar["hour_local"], solar["summer_mw"],
             color=PALETTE["solar"], label="summer (Jun–Aug)")
line_profile(ax, solar["hour_local"], solar["winter_mw"],
             color=PALETTE["muted"], label="winter (Dec–Feb)")
ax.set(title="Solar generation by hour — summer vs winter",
       xlabel="hour (local)", ylabel="avg solar (MW)")
ax.legend()
plt.show()

display(solar.round(0))

In [ ]:
# Cell M (Phase 3): temperature by month — the RQ2 hook
import matplotlib.pyplot as plt
from src.viz import set_house_style, PALETTE, line_profile
set_house_style()

temp = con.sql("""
SELECT
    EXTRACT(month FROM ts_utc AT TIME ZONE 'Europe/Vienna') AS month,
    AVG(temperature_c) AS avg_temp
FROM weather
GROUP BY month
ORDER BY month
""").df()

fig, ax = plt.subplots()
line_profile(ax, temp["month"], temp["avg_temp"], color=PALETTE["temp"])
ax.set(title="Average temperature by month", xlabel="", ylabel="avg temperature (°C)")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(list("JFMAMJJASOND"))
plt.show()

display(temp.round(1))

In [ ]:
# Cell N (Phase 4 prep): Eurostat GHG → tidy annual table
#
# Raw CSV is WIDE (one column per year) and stuffs the full CRF sector hierarchy
# into two units. We want: MIO_T only (= Mt CO2-eq), the top-level sectors + the
# two national totals, reshaped LONG (one row per sector × year).
# UNPIVOT is DuckDB's wide→long — the SQL twin of pandas .melt().

RAW = PROJECT_ROOT / "data" / "raw"
ghg_csv = RAW / "eurostat_ghg_AT.csv"

con.execute(f"""
CREATE OR REPLACE TABLE ghg_emissions AS
WITH raw AS (
    SELECT * FROM read_csv_auto('{ghg_csv}', header = true)
),
-- keep Mt CO2-eq only, and only the sectors we'll actually use
filtered AS (
    SELECT * EXCLUDE ("geo\\TIME_PERIOD")     -- drop the odd combined dim/time column
    FROM raw
    WHERE unit = 'MIO_T'
      AND src_crf IN (
          'TOTX4_MEMO',   -- Total, excl. LULUCF  -> headline emissions
          'TOTXMEMO',     -- Total, incl. LULUCF  -> net (with land sink)
          'CRF1','CRF2','CRF3','CRF4','CRF5','CRF6'
      )
),
-- wide (year columns) -> long (one row per sector × year)
long AS (
    UNPIVOT filtered
    ON COLUMNS(* EXCLUDE (freq, unit, airpol, src_crf))
    INTO NAME year_str VALUE mt_co2e
)
SELECT
    CAST(year_str AS INTEGER) AS year,
    src_crf                   AS sector_code,
    CASE src_crf
        WHEN 'TOTX4_MEMO' THEN 'Total (excl. LULUCF)'
        WHEN 'TOTXMEMO'   THEN 'Total (incl. LULUCF)'
        WHEN 'CRF1' THEN 'Energy'
        WHEN 'CRF2' THEN 'Industrial processes'
        WHEN 'CRF3' THEN 'Agriculture'
        WHEN 'CRF4' THEN 'LULUCF'
        WHEN 'CRF5' THEN 'Waste'
        WHEN 'CRF6' THEN 'Other'
    END                       AS sector,
    mt_co2e
FROM long
WHERE mt_co2e IS NOT NULL
ORDER BY sector_code, year
""")

n = con.sql("SELECT COUNT(*) FROM ghg_emissions").fetchone()[0]  # pyright: ignore[reportOptionalSubscript]
print(f"ghg_emissions: {n} rows")

# Peek: the headline total trajectory
con.sql("""
SELECT year, mt_co2e AS total_excl_lulucf_mt
FROM ghg_emissions
WHERE sector_code = 'TOTX4_MEMO'
ORDER BY year
""").df()

In [ ]:
# Cell O (Phase 3 EDA): GHG emissions — trajectory + sector mix
# First look at the Eurostat data we just loaded. The stacked emitting sectors sum
# to the total (excl. LULUCF); the black line overlays that total as a sanity check
# and traces the trajectory.
# NOTE: LULUCF (CRF4) is a *sink* (negative values) — excluded from the emissions stack.

import matplotlib.pyplot as plt
from src.viz import set_house_style, PALETTE
set_house_style()

# 5 emitting top-level sectors (exclude LULUCF + the totals)
ghg = con.sql("""
SELECT year, sector, mt_co2e
FROM ghg_emissions
WHERE sector_code IN ('CRF1','CRF2','CRF3','CRF5','CRF6')
ORDER BY year
""").df()

# headline total, to overlay
total = con.sql("""
SELECT year, mt_co2e
FROM ghg_emissions
WHERE sector_code = 'TOTX4_MEMO'
ORDER BY year
""").df()

# long → wide for stackplot, ordered largest → smallest for readability
wide  = ghg.pivot(index="year", columns="sector", values="mt_co2e")
order = wide.mean().sort_values(ascending=False).index.tolist()
wide  = wide[order]

sector_colors = {
    "Energy":               PALETTE["accent"],   # coral  — dominant sector
    "Industrial processes": PALETTE["price"],     # purple
    "Agriculture":          PALETTE["wind"],      # teal
    "Waste":                PALETTE["demand"],    # grey
    "Other":                PALETTE["muted"],
}

fig, ax = plt.subplots(figsize=(12, 5))
ax.stackplot(wide.index, [wide[s] for s in order], labels=order,
             colors=[sector_colors[s] for s in order], alpha=0.9)
ax.plot(total["year"], total["mt_co2e"], color="black", lw=1.2,
        label="Total (excl. LULUCF)")
ax.axvline(2005, color=PALETTE["muted"], ls="--", lw=1)   # the target baseline year
ax.set_title("Austria GHG emissions by sector (excl. LULUCF), 1990–2024")
ax.set_ylabel("Mt CO₂-eq")
ax.set_xlim(wide.index.min(), wide.index.max())
ax.legend(loc="lower left", ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

# quick numeric read: total vs the 2005 baseline
base2005   = total.set_index("year").loc[2005, "mt_co2e"]
total_pct  = total.assign(pct_vs_2005=((total["mt_co2e"] / base2005 - 1) * 100).round(1))
display(total_pct.tail(8))

In [ ]:
# Cell P: inspect the EEA Effort Sharing workbook
# EEA xlsx files often carry title/metadata rows above the real header, and may
# split emissions / AEAs / targets across sheets — so we look before we load.
import pandas as pd

EXTERNAL = PROJECT_ROOT / "data" / "external"
esr_xlsx = EXTERNAL / "eea_esr_effort_sharing.xlsx"

xls = pd.ExcelFile(esr_xlsx)                       # needs openpyxl
print("Sheets:", xls.sheet_names, "\n")

for sheet in xls.sheet_names:
    raw = pd.read_excel(xls, sheet_name=sheet, header=None)
    print("=" * 80)
    print(f"SHEET: {sheet!r}   shape = {raw.shape}")
    print("=" * 80)
    with pd.option_context("display.max_columns", None, "display.width", 220):
        print(raw.head(12).to_string())

    # locate Austria so we can see how the country axis is keyed (code? name? row? col?)
    hits = [(int(r), int(c)) for r, c in zip(*raw.apply(
        lambda col: col.astype(str).str.contains("Austria|^AT$", case=False, regex=True, na=False)
    ).to_numpy().nonzero())][:6]
    print("\n  'Austria'/'AT' appears at (row, col):", hits, "\n")

In [ ]:
# Cell Q: load EEA Effort Sharing (non-ETS) emissions → esr_emissions
# Source sheet 'GHG_ESD' — tidy long format (header row 0), already in Mt CO2 eq.
# Austria slice only (RQ6 is Austria-scoped, like owid_energy_at / ghg_emissions).
# The ESD_ESR flag marks the accounting regime: ESD = AR4 GWPs (2013-2020),
# ESR = AR5 GWPs (2021-2030) — kept so RQ6 can see/handle the change at 2021.

PROCESSED = PROJECT_ROOT / "data" / "processed"
DB_PATH = PROCESSED / "austria_energy.duckdb"
esr_xlsx = PROJECT_ROOT / "data" / "external" / "eea_esr_effort_sharing.xlsx"

df_esr = (
    pd.read_excel(esr_xlsx, sheet_name="GHG_ESD")          # header row 0
      .rename(columns={
          "Country":      "country",
          "Year":         "year",
          "ValueNumeric": "mt_co2e",
          "ESD_ESR":      "regime",
          "Unit":         "unit",
          "Data_source":  "data_source",
      })
)

df_esr = df_esr[df_esr["country"] == "Austria"].copy()

# unit is constant — assert it (so a future format change fails loudly), then drop
assert (df_esr["unit"] == "MtCO2 eq").all(), df_esr["unit"].unique()
df_esr["year"]    = df_esr["year"].astype(int)
df_esr["mt_co2e"] = df_esr["mt_co2e"].astype(float)
df_esr = df_esr[["year", "regime", "mt_co2e", "data_source"]].sort_values("year")

con = duckdb.connect(str(DB_PATH))
con.execute("DROP TABLE IF EXISTS esr_emissions")
con.execute("""
CREATE TABLE esr_emissions (
    year         SMALLINT PRIMARY KEY,
    regime       VARCHAR,        -- 'ESD' (AR4, 2013-2020) or 'ESR' (AR5, 2021-2030)
    mt_co2e      DOUBLE,         -- non-ETS Effort Sharing emissions, Mt CO2-eq
    data_source  VARCHAR
)
""")
# BY NAME → column-name-sensitive insert, not positional (your prices-cell preference)
con.execute("INSERT INTO esr_emissions BY NAME SELECT * FROM df_esr")

n = con.sql("SELECT COUNT(*) FROM esr_emissions").fetchone()[0]
print(f"esr_emissions: {n} rows")
con.sql("SELECT * FROM esr_emissions ORDER BY year").df()

In [ ]:
# Cell R: structural EDA of the ESR series
# (i) span + where the accounting regime switches (ESD/AR4 → ESR/AR5)
con.sql("""
SELECT regime,
       MIN(year) AS first_year,
       MAX(year) AS last_year,
       COUNT(*)  AS n_years
FROM esr_emissions
GROUP BY regime
ORDER BY first_year
""").df()

In [ ]:
# Cell R (cont.): baseline, change vs 2005, YoY delta, and
# the ESR slice as a share of the national total (TOTX4_MEMO, excl. LULUCF).
con.sql("""
WITH base AS (                       -- 2005 reference (ESD-basis in this file)
    SELECT mt_co2e AS base_2005
    FROM esr_emissions
    WHERE year = 2005
),
total AS (                           -- national total excl. LULUCF
    SELECT year, mt_co2e AS total_mt
    FROM ghg_emissions
    WHERE sector_code = 'TOTX4_MEMO'
)
SELECT
    e.year,
    e.regime,
    ROUND(e.mt_co2e, 2)                                        AS esr_mt,
    ROUND(100.0 * (e.mt_co2e / b.base_2005 - 1), 1)            AS pct_vs_2005,
    ROUND(e.mt_co2e - LAG(e.mt_co2e) OVER (ORDER BY e.year), 2) AS yoy_delta,
    ROUND(100.0 * e.mt_co2e / t.total_mt, 1)                   AS share_of_total_pct
FROM esr_emissions e
CROSS JOIN base b
LEFT JOIN total t USING (year)
ORDER BY e.year
""").df()

In [ ]:
# Close the connection
con.close()

## Phase 3 complete

EDA covered completeness, distributions, and seasonal structure across the hourly
data (demand, prices, weather, generation), plus a first look at the annual GHG
inventory. Every finding is a hook for its research question.

- **Completeness.** Every hourly bucket present, zero nulls, zero gaps (verified in
  UTC via row-count-vs-expected and a `LAG` gap scan). No imputation needed.
- **Prices → RQ4.** A 2022 regime shift (median ~€224 vs ~€35 in 2019–20), heavily
  right-skewed. Negative prices ~1.2 % of hours (rising to 3.4 % in 2024), clustered at
  midday (solar oversupply) with a smaller overnight mode. The −500 €/MWh floor binds once.
- **Demand → RQ2.** Double-humped day (morning + evening peaks), weekday > weekend,
  winter-peak / summer-trough. Heating-driven — no summer cooling spike (little AC in AT).
- **Temperature → RQ2.** Warm-summer / cold-winter arc (~2 °C Jan, ~22 °C Jul) — a near
  inverse mirror of demand-by-month.
- **Solar → RQ3.** Summer midday avg ~1025 MW vs winter ~320 MW (~3×), the summer bump
  wider. The duck curve is effectively a summer phenomenon — winter solar barely dents net load.
- **Generation mix → RQ1.** Hydro run-of-river is the baseload backbone; gas is the
  flexible price-setter; coal ~phased out (2020); biomass near-constant (must-run). The
  5 nominal/zero fuels are excluded from variability work.
- **GHG → RQ6.** Total (excl. LULUCF) peaked ~93 Mt in 2005, fell to 66.6 Mt in 2024
  (≈ −28 % vs 2005), most of the drop after 2019. Energy (CRF1) dominates and declines most.

**Artifacts.** `src/viz.py` now holds `PALETTE`, `set_house_style()`, `line_profile()`.
The annual `ghg_emissions` table (tidy long, 1990–2024) was added via `UNPIVOT` for RQ6.

**Next — Phase 4:** one notebook per research question (`03_rq1_…` onward), mixing SQL,
`statsmodels`, and `matplotlib`, each ending with a headline visual and a two-sentence finding.